In [ ]:

# ============================================================
# SECTION 1: IMPORT LIBRARIES AND CONFIGURATION
# Filter Comparison: Final Test Evaluation
# ============================================================

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------------------------------------------------
# Device configuration
# ------------------------------------------------------------

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("Selected device:", DEVICE)

# ------------------------------------------------------------
# Project paths
# Works whether notebook is launched from project root
# or from the notebooks folder.
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Please open the notebook from the bus-speckle-seg project."
    )

PREPROCESSED_DIR = (
    PROJECT_ROOT / "data" / "processed" / "preprocessed"
)

DESPECKLED_DIR = (
    PROJECT_ROOT / "data" / "processed" / "despeckled"
)

MODEL_DIR = PROJECT_ROOT / "models"

MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "processed" /
    "preprocessed_manifest.csv"
)

RESULTS_DIR = (
    PROJECT_ROOT / "results" / "filter_comparison"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Model checkpoints
# ------------------------------------------------------------

CHECKPOINTS = {
    "original": MODEL_DIR / "original_seed42_best.pth",
    "gaussian": MODEL_DIR / "gaussian_seed42_best.pth",
    "median": MODEL_DIR / "median_seed42_best.pth",
    "non_local_means": MODEL_DIR / "non_local_means_seed42_best.pth"
}

# ------------------------------------------------------------
# Evaluation configuration
# ------------------------------------------------------------

IMAGE_SIZE = 256
BATCH_SIZE = 4
NUM_WORKERS = 0
THRESHOLD = 0.5

CONDITIONS = [
    "original",
    "gaussian",
    "median",
    "non_local_means"
]

# ------------------------------------------------------------
# Verify important paths
# ------------------------------------------------------------

assert PREPROCESSED_DIR.exists(), "Original dataset not found"
assert DESPECKLED_DIR.exists(), "Filtered dataset directory not found"
assert MODEL_DIR.exists(), "Model directory not found"
assert MANIFEST_PATH.exists(), "Preprocessed manifest not found"

for condition, checkpoint_path in CHECKPOINTS.items():
    assert checkpoint_path.exists(), (
        f"Missing checkpoint for {condition}: {checkpoint_path}"
    )

# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("\n========== PROJECT CONFIGURATION ==========")

print("Project root:", PROJECT_ROOT)
print("Original dataset:", PREPROCESSED_DIR)
print("Filtered datasets:", DESPECKLED_DIR)
print("Model directory:", MODEL_DIR)
print("Results directory:", RESULTS_DIR)

print("\nEvaluation conditions:")
for condition, path in CHECKPOINTS.items():
    print(f"{condition:<18} {path.name}")

print("\nImage size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Prediction threshold:", THRESHOLD)

print("\nAll required paths and checkpoints verified.")


In [ ]:

# ============================================================
# SECTION 2: TEST DATASET AND DATALOADERS
# ============================================================

# ------------------------------------------------------------
# 1. Load preprocessed dataset metadata
# ------------------------------------------------------------

manifest_df = pd.read_csv(MANIFEST_PATH)

test_manifest = manifest_df[
    manifest_df["split"] == "test"
].copy()

# Map processed image filename to its category
test_manifest["image_name"] = test_manifest["image_path"].apply(
    lambda x: Path(x).name
)

# Create metadata lookup
test_metadata = test_manifest.set_index("image_name")["category"].to_dict()

print("Test metadata loaded.")
print("Total test images:", len(test_manifest))

print("\nCategory distribution:")
print(test_manifest["category"].value_counts())


# ------------------------------------------------------------
# 2. Custom dataset for each filtering condition
# ------------------------------------------------------------

class FilterComparisonDataset(Dataset):

    def __init__(self, condition_root):

        self.condition_root = Path(condition_root)

        # Images come from the selected filtering condition
        self.image_dir = (
            self.condition_root / "test" / "images"
        )

        # Always use original ground-truth masks
        self.mask_dir = (
            PREPROCESSED_DIR / "test" / "masks"
        )

        self.image_paths = sorted(
            self.image_dir.glob("*.png")
        )

        self.mask_paths = [
            self.mask_dir / image_path.name
            for image_path in self.image_paths
        ]

        if len(self.image_paths) == 0:
            raise FileNotFoundError(
                f"No test images found: {self.image_dir}"
            )

        # Verify image-mask correspondence
        for image_path, mask_path in zip(
            self.image_paths, self.mask_paths
        ):

            if not mask_path.exists():
                raise FileNotFoundError(
                    f"Missing ground-truth mask: {mask_path}"
                )

        # Verify metadata correspondence
        image_names = [
            path.name for path in self.image_paths
        ]

        missing_metadata = [
            name for name in image_names
            if name not in test_metadata
        ]

        if missing_metadata:
            raise ValueError(
                f"Missing category metadata: {missing_metadata}"
            )

        # Verify every condition has the same test images
        original_names = sorted(
            p.name for p in (
                PREPROCESSED_DIR / "test" / "images"
            ).glob("*.png")
        )

        if sorted(image_names) != original_names:
            raise ValueError(
                "Test image filenames differ from the original dataset."
            )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):

        image_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        # Load image
        image = Image.open(image_path).convert("L")

        # Load original ground-truth mask
        mask = Image.open(mask_path).convert("L")

        image = np.array(image, dtype=np.float32) / 255.0
        mask = np.array(mask, dtype=np.float32)

        # Binarize mask
        mask = (mask > 0).astype(np.float32)

        # Verify dimensions
        if image.shape != (IMAGE_SIZE, IMAGE_SIZE):
            raise ValueError(
                f"Unexpected image shape: {image_path.name}"
            )

        if mask.shape != (IMAGE_SIZE, IMAGE_SIZE):
            raise ValueError(
                f"Unexpected mask shape: {mask_path.name}"
            )

        image_tensor = torch.from_numpy(image).unsqueeze(0)
        mask_tensor = torch.from_numpy(mask).unsqueeze(0)

        filename = image_path.name
        category = test_metadata[filename]

        return image_tensor, mask_tensor, filename, category


# ------------------------------------------------------------
# 3. Create datasets and DataLoaders
# ------------------------------------------------------------

test_datasets = {}
test_loaders = {}

for condition in CONDITIONS:

    if condition == "original":
        condition_root = PREPROCESSED_DIR

    else:
        condition_root = DESPECKLED_DIR / condition

    dataset = FilterComparisonDataset(condition_root)

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS
    )

    test_datasets[condition] = dataset
    test_loaders[condition] = loader

    print(
        f"{condition:<18} "
        f"Images: {len(dataset)} | "
        f"Batches: {len(loader)}"
    )


# ------------------------------------------------------------
# 4. Verify category distribution for all conditions
# ------------------------------------------------------------

print("\n========== CATEGORY VERIFICATION ==========")

for condition, dataset in test_datasets.items():

    categories = pd.Series([
        test_metadata[path.name]
        for path in dataset.image_paths
    ])

    print(f"\n{condition.upper()}")
    print(categories.value_counts().to_dict())


# ------------------------------------------------------------
# 5. Verify a sample batch
# ------------------------------------------------------------

sample_images, sample_masks, sample_names, sample_categories = (
    next(iter(test_loaders["original"]))
)

print("\n========== BATCH VERIFICATION ==========")

print("Image tensor shape:", sample_images.shape)
print("Mask tensor shape:", sample_masks.shape)

print(
    "Image value range:",
    sample_images.min().item(),
    "to",
    sample_images.max().item()
)

print("Mask unique values:", torch.unique(sample_masks))

print("Sample filenames:", list(sample_names))
print("Sample categories:", list(sample_categories))


# ------------------------------------------------------------
# 6. Final assertions
# ------------------------------------------------------------

assert len(test_manifest) == 118

assert test_manifest["category"].value_counts().to_dict() == {
    "benign": 67,
    "malignant": 30,
    "normal": 21
}

assert all(
    len(dataset) == 118
    for dataset in test_datasets.values()
)

print("\nAll test datasets loaded and verified successfully.")


In [ ]:

# ============================================================
# SECTION 3: U-NET ARCHITECTURE AND CHECKPOINT LOADING
# ============================================================

# ------------------------------------------------------------
# 1. Define DoubleConv block
# ------------------------------------------------------------

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


# ------------------------------------------------------------
# 2. Define exact U-Net architecture used during training
# ------------------------------------------------------------

class UNet(nn.Module):

    def __init__(self, in_channels=1, out_channels=1):

        super().__init__()

        # Encoder
        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)

        # Bottleneck
        self.bottleneck = DoubleConv(128, 256)

        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2)

        # Decoder
        self.up3 = nn.ConvTranspose2d(
            256, 128, kernel_size=2, stride=2
        )
        self.dec3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(
            128, 64, kernel_size=2, stride=2
        )
        self.dec2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(
            64, 32, kernel_size=2, stride=2
        )
        self.dec1 = DoubleConv(64, 32)

        # Output layer
        self.output = nn.Conv2d(
            32, out_channels, kernel_size=1
        )

    def forward(self, x):

        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        # Bottleneck
        b = self.bottleneck(self.pool(e3))

        # Decoder with skip connections
        d3 = self.up3(b)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        # Raw logits
        return self.output(d1)


# ------------------------------------------------------------
# 3. Verify model architecture
# ------------------------------------------------------------

reference_model = UNet(
    in_channels=1,
    out_channels=1
)

total_params = sum(
    parameter.numel()
    for parameter in reference_model.parameters()
)

print("U-Net architecture initialized.")
print("Total parameters:", f"{total_params:,}")

assert total_params == 1_926_433, (
    "Architecture does not match the training model."
)

del reference_model


# ------------------------------------------------------------
# 4. Load all four best checkpoints
# ------------------------------------------------------------

models = {}
checkpoint_info = []

for condition in CONDITIONS:

    print(f"\nLoading {condition} model...")

    # Initialize a fresh model
    model = UNet(
        in_channels=1,
        out_channels=1
    ).to(DEVICE)

    # Load checkpoint
    checkpoint = torch.load(
        CHECKPOINTS[condition],
        map_location=DEVICE,
        weights_only=False
    )

    # Load trained weights
    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    # Set evaluation mode
    model.eval()

    # Keep model available for prediction
    models[condition] = model

    # Store checkpoint information
    checkpoint_info.append({
        "condition": condition,
        "checkpoint": CHECKPOINTS[condition].name,
        "best_epoch": checkpoint["epoch"],
        "best_validation_dice": checkpoint["best_val_dice"],
        "seed": checkpoint["config"]["seed"]
    })

    print("Checkpoint loaded successfully.")
    print("Best epoch:", checkpoint["epoch"])
    print(
        "Best validation Dice:",
        f"{checkpoint['best_val_dice']:.4f}"
    )


# ------------------------------------------------------------
# 5. Display checkpoint summary
# ------------------------------------------------------------

checkpoint_summary = pd.DataFrame(checkpoint_info)

print("\n========== CHECKPOINT SUMMARY ==========")

display(checkpoint_summary)

print("\nModels loaded:", len(models))
print("Model conditions:", list(models.keys()))


# ------------------------------------------------------------
# 6. Verify forward pass for all four models
# ------------------------------------------------------------

print("\n========== FORWARD PASS VERIFICATION ==========")

sample_input = sample_images[:1].to(DEVICE)

with torch.no_grad():

    for condition, model in models.items():

        output = model(sample_input)

        print(
            f"{condition:<18} "
            f"Input: {tuple(sample_input.shape)} | "
            f"Output: {tuple(output.shape)}"
        )

        assert output.shape == (
            1, 1, IMAGE_SIZE, IMAGE_SIZE
        )

print("\nAll four U-Net models loaded and verified successfully.")


## 4. Test Set Evaluation

Evaluate all four trained U-Net models on the same 118 test images.

Metrics:
- Dice coefficient
- Intersection over Union (IoU)
- Pixel-level precision
- Pixel-level sensitivity
- Category-wise lesion segmentation performance
- False positives on normal images

All predictions use a fixed probability threshold of 0.5.

In [ ]:
def calculate_metrics(prediction, ground_truth):
    """
    Calculate Dice, IoU, precision, and sensitivity
    for a single binary prediction and ground-truth mask.
    """

    prediction = prediction.astype(bool)
    ground_truth = ground_truth.astype(bool)

    tp = np.logical_and(prediction, ground_truth).sum()
    fp = np.logical_and(prediction, ~ground_truth).sum()
    fn = np.logical_and(~prediction, ground_truth).sum()

    dice = (2 * tp) / (2 * tp + fp + fn + 1e-8)

    iou = tp / (tp + fp + fn + 1e-8)

    precision = tp / (tp + fp + 1e-8)

    sensitivity = tp / (tp + fn + 1e-8)

    return {
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "sensitivity": sensitivity,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn)
    }


print("Evaluation metric function defined successfully.")

In [ ]:
# =====================================================
# SECTION 4.2: TEST SET EVALUATION
# =====================================================

all_results = []

for condition in models.keys():

    print(f"\nEvaluating {condition} model...")

    model = models[condition]
    loader = test_loaders[condition]

    model.eval()

    with torch.no_grad():

        for images, masks, filenames, categories in loader:

            images = images.to(DEVICE)

            logits = model(images)
            probabilities = torch.sigmoid(logits)

            predictions = (probabilities >= THRESHOLD).float()

            predictions = predictions.cpu().numpy()
            masks = masks.cpu().numpy()

            for i in range(len(filenames)):

                prediction = predictions[i, 0]
                ground_truth = masks[i, 0]

                metrics = calculate_metrics(
                    prediction,
                    ground_truth
                )

                category = categories[i]

                # Predicted positive pixels
                predicted_area = int(prediction.sum())

                # Ground-truth positive pixels
                ground_truth_area = int(ground_truth.sum())

                result = {
                    "condition": condition,
                    "filename": filenames[i],
                    "category": category,
                    **metrics,
                    "predicted_area": predicted_area,
                    "ground_truth_area": ground_truth_area,
                    "is_normal": category == "normal",
                    "false_positive_image": (
                        category == "normal" and predicted_area > 0
                    )
                }

                all_results.append(result)

    print(
        f"Completed: {sum(r['condition'] == condition for r in all_results)} "
        f"images evaluated"
    )


# Convert results into a DataFrame
results_df = pd.DataFrame(all_results)

print("\n========== EVALUATION COMPLETED ==========")

print("Total predictions:", len(results_df))
print("\nPredictions per condition:")
print(results_df["condition"].value_counts())

print("\nCategory distribution:")
print(results_df.groupby(
    ["condition", "category"]
).size())

print("\nResults DataFrame shape:", results_df.shape)

results_df.head()

In [ ]:
# =====================================================
# SECTION 4.3: SUMMARY METRICS
# =====================================================

# Exclude normal images from lesion segmentation metrics
lesion_results = results_df[
    results_df["category"].isin(["benign", "malignant"])
].copy()

normal_results = results_df[
    results_df["category"] == "normal"
].copy()


# -----------------------------------------------------
# 1. OVERALL LESION SEGMENTATION PERFORMANCE
# -----------------------------------------------------

overall_summary = []

for condition in models.keys():

    subset = lesion_results[
        lesion_results["condition"] == condition
    ]

    total_tp = subset["tp"].sum()
    total_fp = subset["fp"].sum()
    total_fn = subset["fn"].sum()

    overall_summary.append({
        "condition": condition,
        "lesion_images": len(subset),

        # Equal-weighted mean across images
        "mean_dice": subset["dice"].mean(),
        "mean_iou": subset["iou"].mean(),

        # Pooled pixel-level metrics
        "pixel_precision": total_tp / (total_tp + total_fp + 1e-8),
        "pixel_sensitivity": total_tp / (total_tp + total_fn + 1e-8),

        "total_tp": total_tp,
        "total_fp": total_fp,
        "total_fn": total_fn
    })


overall_summary_df = pd.DataFrame(overall_summary)


print("========== OVERALL LESION PERFORMANCE ==========")

display(
    overall_summary_df.round(4)
)


# -----------------------------------------------------
# 2. CATEGORY-WISE PERFORMANCE
# -----------------------------------------------------

category_summary = []

for condition in models.keys():

    for category in ["benign", "malignant"]:

        subset = lesion_results[
            (lesion_results["condition"] == condition) &
            (lesion_results["category"] == category)
        ]

        category_summary.append({
            "condition": condition,
            "category": category,
            "images": len(subset),
            "mean_dice": subset["dice"].mean(),
            "mean_iou": subset["iou"].mean()
        })


category_summary_df = pd.DataFrame(category_summary)


print("\n========== CATEGORY-WISE PERFORMANCE ==========")

display(
    category_summary_df.round(4)
)


# -----------------------------------------------------
# 3. NORMAL IMAGE FALSE POSITIVES
# -----------------------------------------------------

normal_summary = []

for condition in models.keys():

    subset = normal_results[
        normal_results["condition"] == condition
    ]

    false_positive_count = subset["false_positive_image"].sum()

    normal_summary.append({
        "condition": condition,
        "normal_images": len(subset),
        "false_positive_images": int(false_positive_count),
        "false_positive_rate": (
            false_positive_count / len(subset)
        ),
        "mean_predicted_area": subset["predicted_area"].mean(),
        "total_predicted_area": subset["predicted_area"].sum()
    })


normal_summary_df = pd.DataFrame(normal_summary)


print("\n========== NORMAL IMAGE PERFORMANCE ==========")

display(
    normal_summary_df.round(4)
)

In [ ]:
# =====================================================
# SECTION 4.4: SAVE EVALUATION RESULTS
# =====================================================

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save per-image metrics
results_df.to_csv(
    RESULTS_DIR / "per_image_metrics.csv",
    index=False
)

# Save summary tables
overall_summary_df.to_csv(
    RESULTS_DIR / "overall_comparison.csv",
    index=False
)

category_summary_df.to_csv(
    RESULTS_DIR / "category_comparison.csv",
    index=False
)

normal_summary_df.to_csv(
    RESULTS_DIR / "normal_image_comparison.csv",
    index=False
)

print("Evaluation results saved successfully.")

print("\nSaved files:")
for filename in [
    "per_image_metrics.csv",
    "overall_comparison.csv",
    "category_comparison.csv",
    "normal_image_comparison.csv"
]:
    print("-", RESULTS_DIR / filename)

In [ ]:
# =====================================================
# SECTION 5.1: INSPECT PREVIOUS EVALUATION
# =====================================================

import json

evaluation_notebooks = list(
    PROJECT_ROOT.rglob("04_model_evaluation.ipynb")
)

print("Matching notebooks:", len(evaluation_notebooks))

for path in evaluation_notebooks:
    print(path)

if len(evaluation_notebooks) == 1:

    old_notebook_path = evaluation_notebooks[0]

    with open(old_notebook_path, "r", encoding="utf-8") as file:
        old_notebook = json.load(file)

    print("\n========== RELEVANT CELLS FROM OLD NOTEBOOK ==========")

    keywords = [
        "dice",
        "iou",
        "precision",
        "sensitivity",
        "threshold",
        "checkpoint",
        "test_loader",
        "test_dataset"
    ]

    for i, cell in enumerate(old_notebook["cells"]):

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])
        source_lower = source.lower()

        if (
            any(word in source_lower for word in keywords)
            and (
                "test" in source_lower
                or "dice" in source_lower
                or "checkpoint" in source_lower
            )
        ):
            print(f"\n--- Cell {i} ---")
            print(source)

elif len(evaluation_notebooks) == 0:
    print("Could not find 04_model_evaluation.ipynb.")

else:
    print("Multiple matching notebooks found. We need to identify the correct one.")

In [ ]:
# =====================================================
# SECTION 5.2: CHECKPOINT AND BASELINE COMPARISON
# =====================================================

baseline_checkpoint_path = MODEL_DIR / "baseline_unet_best.pth"
original_checkpoint_path = MODEL_DIR / "original_seed42_best.pth"

print("========== CHECKPOINT VERIFICATION ==========")

print("Baseline checkpoint exists:", baseline_checkpoint_path.exists())
print("Comparison checkpoint exists:", original_checkpoint_path.exists())

if baseline_checkpoint_path.exists() and original_checkpoint_path.exists():

    baseline_ckpt = torch.load(
        baseline_checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    original_ckpt = torch.load(
        original_checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    baseline_weights = baseline_ckpt["model_state_dict"]
    original_weights = original_ckpt["model_state_dict"]

    print("\nBaseline checkpoint:")
    print("Epoch:", baseline_ckpt.get("epoch"))
    print("Best validation Dice:", baseline_ckpt.get("best_val_dice"))

    print("\nOriginal comparison checkpoint:")
    print("Epoch:", original_ckpt.get("epoch"))
    print("Best validation Dice:", original_ckpt.get("best_val_dice"))

    same_keys = baseline_weights.keys() == original_weights.keys()

    print("\nSame model parameter keys:", same_keys)

    if same_keys:

        total_tensors = len(baseline_weights)
        identical_tensors = 0
        max_difference = 0.0

        for key in baseline_weights:

            weight_a = baseline_weights[key]
            weight_b = original_weights[key]

            if torch.equal(weight_a, weight_b):
                identical_tensors += 1

            difference = (weight_a - weight_b).abs().max().item()
            max_difference = max(max_difference, difference)

        print("Total parameter tensors:", total_tensors)
        print("Identical parameter tensors:", identical_tensors)
        print("Maximum absolute weight difference:", max_difference)

        print(
            "Are checkpoints identical:",
            identical_tensors == total_tensors
        )


# -----------------------------------------------------
# Compare old and current per-image evaluation results
# -----------------------------------------------------

old_csv_path = (
    PROJECT_ROOT / "results" / "evaluation" /
    "baseline_test_results.csv"
)

new_csv_path = RESULTS_DIR / "per_image_metrics.csv"

print("\n========== PER-IMAGE RESULT COMPARISON ==========")

print("Old results CSV exists:", old_csv_path.exists())
print("New results CSV exists:", new_csv_path.exists())

if old_csv_path.exists() and new_csv_path.exists():

    old_df = pd.read_csv(old_csv_path)
    new_df = pd.read_csv(new_csv_path)

    # Preserve the original filenames, not category-stripped names
    if "image_name" in old_df.columns:
        old_df = old_df.rename(columns={"image_name": "filename"})

    old_columns = [
        "filename", "dice", "iou", "tp", "fp", "fn"
    ]

    new_columns = [
        "filename", "dice", "iou", "tp", "fp", "fn"
    ]

    old_compare = old_df[old_columns].copy()
    new_compare = new_df[
        new_df["condition"] == "original"
    ][new_columns].copy()

    merged = old_compare.merge(
        new_compare,
        on="filename",
        suffixes=("_baseline", "_comparison"),
        how="inner",
        validate="one_to_one"
    )

    print("Matched images:", len(merged))

    # Identify lesion images using ground-truth pixel counts
    if "target_pixels" in old_df.columns:

        lesion_filenames = set(
            old_df.loc[
                old_df["target_pixels"] > 0,
                "filename"
            ]
        )

        matched_lesions = merged[
            merged["filename"].isin(lesion_filenames)
        ].copy()

    else:
        # All 97 lesion images are category-labeled
        matched_lesions = merged[
            merged["filename"].isin(
                results_df.loc[
                    results_df["category"] != "normal",
                    "filename"
                ].unique()
            )
        ].copy()

    print("Matched lesion images:", len(matched_lesions))

    print("\nOld baseline mean Dice:",
          matched_lesions["dice_baseline"].mean())

    print("Current original mean Dice:",
          matched_lesions["dice_comparison"].mean())

    print(
        "Mean absolute per-image Dice difference:",
        (
            matched_lesions["dice_baseline"] -
            matched_lesions["dice_comparison"]
        ).abs().mean()
    )

else:
    print("One or both CSV files are unavailable for comparison.")

In [ ]:
# =====================================================
# SECTION 5.3: CORRECTED BASELINE COMPARISON
# =====================================================

old_df = pd.read_csv(old_csv_path)
new_df = pd.read_csv(new_csv_path)

# Use image_name as the original filename in the old CSV.
# The old CSV also contains a category-stripped filename column.
old_compare = old_df[
    ["image_name", "dice", "iou", "tp", "fp", "fn"]
].rename(columns={"image_name": "filename"})

new_compare = new_df[
    new_df["condition"] == "original"
][["filename", "dice", "iou", "tp", "fp", "fn"]]

# Verify unique filenames
assert old_compare["filename"].is_unique
assert new_compare["filename"].is_unique

# Match the same test images
merged = old_compare.merge(
    new_compare,
    on="filename",
    suffixes=("_baseline", "_comparison"),
    how="inner",
    validate="one_to_one"
)

print("========== CORRECTED COMPARISON ==========")

print("Matched test images:", len(merged))

# Identify lesion images using original ground-truth pixel counts
lesion_filenames = set(
    old_df.loc[
        old_df["target_pixels"] > 0,
        "image_name"
    ]
)

matched_lesions = merged[
    merged["filename"].isin(lesion_filenames)
].copy()

print("Matched lesion images:", len(matched_lesions))

print("\nOld baseline mean Dice:",
      matched_lesions["dice_baseline"].mean())

print("Current original mean Dice:",
      matched_lesions["dice_comparison"].mean())

print("\nMean per-image metric differences:")

for metric in ["dice", "iou"]:
    difference = (
        matched_lesions[f"{metric}_comparison"] -
        matched_lesions[f"{metric}_baseline"]
    )

    print(f"{metric.upper()} difference: {difference.mean():.4f}")

print("\nImages where current model performed better:",
      (matched_lesions["dice_comparison"] >
       matched_lesions["dice_baseline"]).sum())

print("Images where earlier baseline performed better:",
      (matched_lesions["dice_baseline"] >
       matched_lesions["dice_comparison"]).sum())

assert len(merged) == 118
assert len(matched_lesions) == 97

print("\nBoth evaluations matched successfully.")

## 6. Statistical Comparison

Compare each despeckling method against the original U-Net using
paired per-image Dice and IoU values.

Methods:
- Paired bootstrap confidence intervals (95%)
- Wilcoxon signed-rank tests
- Holm correction for multiple comparisons

Only the 97 lesion images are included in this analysis.

In [ ]:
# =====================================================
# SECTION 6.1: PAIRED STATISTICAL COMPARISON
# =====================================================

from scipy.stats import wilcoxon

# -----------------------------------------------------
# 1. Prepare paired lesion-level data
# -----------------------------------------------------

lesion_results = results_df[
    results_df["category"].isin(["benign", "malignant"])
].copy()

conditions = [
    "original",
    "gaussian",
    "median",
    "non_local_means"
]

# Pivot so each row represents the same image
dice_pivot = lesion_results.pivot(
    index="filename",
    columns="condition",
    values="dice"
)

iou_pivot = lesion_results.pivot(
    index="filename",
    columns="condition",
    values="iou"
)

# Verify matching observations
assert len(dice_pivot) == 97
assert len(iou_pivot) == 97

assert set(dice_pivot.columns) == set(conditions)
assert set(iou_pivot.columns) == set(conditions)

assert not dice_pivot.isna().any().any()
assert not iou_pivot.isna().any().any()

print("Paired lesion images:", len(dice_pivot))
print("Conditions:", list(dice_pivot.columns))


# -----------------------------------------------------
# 2. Paired bootstrap confidence intervals
# -----------------------------------------------------

rng = np.random.default_rng(42)

N_BOOTSTRAP = 10000
N_IMAGES = len(dice_pivot)

# Same sampled image indices are used within each
# comparison, preserving the paired structure.
bootstrap_indices = rng.integers(
    0,
    N_IMAGES,
    size=(N_BOOTSTRAP, N_IMAGES)
)


def paired_bootstrap_ci(differences):

    differences = np.asarray(differences, dtype=float)

    bootstrap_means = differences[
        bootstrap_indices
    ].mean(axis=1)

    lower = np.percentile(bootstrap_means, 2.5)
    upper = np.percentile(bootstrap_means, 97.5)

    return lower, upper


# -----------------------------------------------------
# 3. Wilcoxon tests and effect estimates
# -----------------------------------------------------

statistical_rows = []

for condition in ["gaussian", "median", "non_local_means"]:

    for metric, pivot in [
        ("dice", dice_pivot),
        ("iou", iou_pivot)
    ]:

        # Positive difference = filtered model is better
        differences = (
            pivot[condition] - pivot["original"]
        ).to_numpy()

        mean_difference = differences.mean()

        ci_lower, ci_upper = paired_bootstrap_ci(
            differences
        )

        test_result = wilcoxon(
            differences,
            alternative="two-sided",
            zero_method="wilcox",
            method="auto"
        )

        statistical_rows.append({
            "condition": condition,
            "metric": metric,
            "mean_difference": mean_difference,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
            "wilcoxon_statistic": test_result.statistic,
            "p_value": test_result.pvalue
        })


statistical_summary_df = pd.DataFrame(statistical_rows)


# -----------------------------------------------------
# 4. Holm correction
# -----------------------------------------------------

def holm_adjust(p_values):

    p_values = np.asarray(p_values, dtype=float)

    n = len(p_values)
    order = np.argsort(p_values)

    sorted_p = p_values[order]

    adjusted_sorted = np.maximum.accumulate(
        (n - np.arange(n)) * sorted_p
    )

    adjusted_sorted = np.minimum(adjusted_sorted, 1.0)

    adjusted = np.empty(n)
    adjusted[order] = adjusted_sorted

    return adjusted


# Adjust Dice and IoU comparisons separately
for metric in ["dice", "iou"]:

    mask = statistical_summary_df["metric"] == metric

    statistical_summary_df.loc[
        mask, "p_value_holm"
    ] = holm_adjust(
        statistical_summary_df.loc[
            mask, "p_value"
        ].to_numpy()
    )


# -----------------------------------------------------
# 5. Display results
# -----------------------------------------------------

print("\n========== PAIRED STATISTICAL RESULTS ==========")

display(
    statistical_summary_df.round(5)
)

print("\nInterpretation:")
print("Positive mean difference = filtered model performed better.")
print("Negative mean difference = original model performed better.")
print("95% CI is calculated using paired bootstrap resampling.")
print("Holm-adjusted p-values account for three comparisons per metric.")

In [ ]:
# =====================================================
# SECTION 6.2: SAVE STATISTICAL RESULTS
# =====================================================

# Save statistical comparison table
statistical_summary_df.to_csv(
    RESULTS_DIR / "statistical_comparison.csv",
    index=False
)

# Calculate the Dice change for each individual lesion image
per_image_dice_change = dice_pivot.copy()

per_image_dice_change["mean_filtered_dice"] = (
    per_image_dice_change[
        ["gaussian", "median", "non_local_means"]
    ].mean(axis=1)
)

per_image_dice_change["mean_filtered_change"] = (
    per_image_dice_change["mean_filtered_dice"]
    - per_image_dice_change["original"]
)

# Add category information
category_lookup = (
    lesion_results[["filename", "category"]]
    .drop_duplicates()
    .set_index("filename")["category"]
)

per_image_dice_change["category"] = category_lookup

# Save per-image paired differences
per_image_dice_change.to_csv(
    RESULTS_DIR / "per_image_dice_differences.csv"
)

print("Statistical results saved successfully.")

print("\nSaved files:")
print("- statistical_comparison.csv")
print("- per_image_dice_differences.csv")

print("\nMean filtered Dice improvement across individual images:",
      round(per_image_dice_change["mean_filtered_change"].mean(), 5))

print("\nLargest improvement across filters:")
print(per_image_dice_change["mean_filtered_change"].idxmax())

print("\nLargest decline across filters:")
print(per_image_dice_change["mean_filtered_change"].idxmin())

## 7. Prediction Visualizations

Visual comparison of original and despeckled U-Net predictions.

Three representative cases:
- Largest average improvement
- Typical improvement
- Largest average decline

In [ ]:
# =====================================================
# SECTION 7.1: SELECT REPRESENTATIVE IMAGES
# =====================================================

# Largest average improvement across the three filters
best_case = per_image_dice_change[
    "mean_filtered_change"
].idxmax()

# Largest average decline across the three filters
worst_case = per_image_dice_change[
    "mean_filtered_change"
].idxmin()

# Typical case: closest to the median average improvement
median_change = per_image_dice_change[
    "mean_filtered_change"
].median()

typical_case = (
    per_image_dice_change["mean_filtered_change"]
    .sub(median_change)
    .abs()
    .idxmin()
)

selected_cases = {
    "Largest improvement": best_case,
    "Typical case": typical_case,
    "Largest decline": worst_case
}


# -----------------------------------------------------
# Display selected images and their individual metrics
# -----------------------------------------------------

selected_rows = []

for label, filename in selected_cases.items():

    row = per_image_dice_change.loc[filename]

    selected_rows.append({
        "case": label,
        "filename": filename,
        "category": row["category"],
        "original_dice": row["original"],
        "gaussian_dice": row["gaussian"],
        "median_dice": row["median"],
        "nlm_dice": row["non_local_means"],
        "mean_filtered_change": row["mean_filtered_change"]
    })


selected_cases_df = pd.DataFrame(selected_rows)

print("========== SELECTED VISUALIZATION CASES ==========")

display(selected_cases_df.round(4))


# Save selected cases
selected_cases_df.to_csv(
    RESULTS_DIR / "representative_visualization_cases.csv",
    index=False
)

print("\nRepresentative cases saved successfully.")

In [ ]:
# =====================================================
# SECTION 7.2: RETRIEVE REPRESENTATIVE PREDICTIONS
# =====================================================

comparison_conditions = [
    "original",
    "gaussian",
    "median",
    "non_local_means"
]

# Prepare storage for the selected images
selected_visuals = {
    filename: {
        "category": None,
        "inputs": {},
        "predictions": {},
        "ground_truth": None
    }
    for filename in selected_cases.values()
}


# -----------------------------------------------------
# Retrieve each selected image and its prediction
# -----------------------------------------------------

for condition in comparison_conditions:

    print(f"\nRetrieving predictions: {condition}")

    model = models[condition]
    loader = test_loaders[condition]

    model.eval()

    with torch.no_grad():

        for images, masks, filenames, categories in loader:

            # Find selected images in this batch
            selected_indices = [
                i for i, filename in enumerate(filenames)
                if filename in selected_visuals
            ]

            if not selected_indices:
                continue

            # Run inference only on selected images
            selected_batch = images[selected_indices].to(DEVICE)

            logits = model(selected_batch)
            probabilities = torch.sigmoid(logits)

            predictions = (
                probabilities >= THRESHOLD
            ).cpu().numpy()[:, 0].astype(bool)

            # Store the images and predictions
            for output_idx, batch_idx in enumerate(selected_indices):

                filename = filenames[batch_idx]
                entry = selected_visuals[filename]

                input_image = images[batch_idx, 0].numpy()
                ground_truth = masks[batch_idx, 0].numpy().astype(bool)
                prediction = predictions[output_idx]

                entry["category"] = categories[batch_idx]
                entry["inputs"][condition] = input_image
                entry["predictions"][condition] = prediction

                # Verify that ground truth is identical
                # across all four conditions
                if entry["ground_truth"] is None:
                    entry["ground_truth"] = ground_truth
                else:
                    assert np.array_equal(
                        entry["ground_truth"],
                        ground_truth
                    ), f"Ground truth mismatch: {filename}"


# -----------------------------------------------------
# Verify that all selected images were retrieved
# -----------------------------------------------------

for filename, entry in selected_visuals.items():

    assert set(entry["inputs"].keys()) == set(comparison_conditions)
    assert set(entry["predictions"].keys()) == set(comparison_conditions)

    assert entry["ground_truth"] is not None

    for condition in comparison_conditions:
        assert entry["inputs"][condition].shape == (256, 256)
        assert entry["predictions"][condition].shape == (256, 256)

print("\n========== VISUALIZATION DATA VERIFICATION ==========")

for filename, entry in selected_visuals.items():

    print(f"\nImage: {filename}")
    print("Category:", entry["category"])
    print("Ground truth lesion pixels:", int(entry["ground_truth"].sum()))

    for condition in comparison_conditions:
        print(
            f"{condition:18s} | "
            f"Predicted pixels: "
            f"{int(entry['predictions'][condition].sum())}"
        )

print("\nAll representative images and predictions retrieved successfully.")

In [ ]:
# =====================================================
# SECTION 7.3: GENERATE REPRESENTATIVE VISUALIZATIONS
# =====================================================

from matplotlib.patches import Patch
from matplotlib.lines import Line2D

VISUALIZATION_DIR = RESULTS_DIR / "visualizations"
VISUALIZATION_DIR.mkdir(parents=True, exist_ok=True)

comparison_conditions = [
    "original",
    "gaussian",
    "median",
    "non_local_means"
]

condition_titles = {
    "original": "Original",
    "gaussian": "Gaussian",
    "median": "Median",
    "non_local_means": "Non-Local Means"
}

saved_figures = []


# -----------------------------------------------------
# Generate one figure for each representative image
# -----------------------------------------------------

for case_label, filename in selected_cases.items():

    entry = selected_visuals[filename]

    original_image = entry["inputs"]["original"]
    ground_truth = entry["ground_truth"]

    fig, axes = plt.subplots(
        2, 5,
        figsize=(19, 8)
    )

    fig.suptitle(
        f"{case_label}: {filename} ({entry['category']})",
        fontsize=15,
        fontweight="bold"
    )

    # -------------------------------------------------
    # TOP ROW: INPUT IMAGES AND GROUND-TRUTH MASK
    # -------------------------------------------------

    for col, condition in enumerate(comparison_conditions):

        ax = axes[0, col]

        ax.imshow(
            entry["inputs"][condition],
            cmap="gray",
            vmin=0,
            vmax=1
        )

        ax.set_title(
            condition_titles[condition],
            fontsize=11
        )

        ax.axis("off")

    # Ground-truth mask in the fifth column
    axes[0, 4].imshow(
        ground_truth,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[0, 4].set_title("Ground Truth Mask", fontsize=11)
    axes[0, 4].axis("off")

    axes[0, 0].set_ylabel(
        "INPUTS",
        fontsize=12,
        fontweight="bold"
    )


    # -------------------------------------------------
    # BOTTOM ROW: PREDICTIONS ON THE SAME ORIGINAL IMAGE
    # -------------------------------------------------

    for col, condition in enumerate(comparison_conditions):

        ax = axes[1, col]

        prediction = entry["predictions"][condition]

        # Always use the same original ultrasound background
        ax.imshow(
            original_image,
            cmap="gray",
            vmin=0,
            vmax=1
        )

        # Red translucent predicted region
        prediction_overlay = np.zeros(
            (*prediction.shape, 4),
            dtype=np.float32
        )

        prediction_overlay[prediction] = [1, 0, 0, 0.40]

        ax.imshow(prediction_overlay)

        # Green boundary represents ground truth
        if ground_truth.any():

            ax.contour(
                ground_truth.astype(float),
                levels=[0.5],
                colors="lime",
                linewidths=1.2
            )

        # Retrieve Dice for this particular image and model
        dice_score = results_df.loc[
            (results_df["filename"] == filename) &
            (results_df["condition"] == condition),
            "dice"
        ].iloc[0]

        ax.set_title(
            f"{condition_titles[condition]}\nDice: {dice_score:.4f}",
            fontsize=10
        )

        ax.axis("off")


    # Ground-truth overlay in the fifth column
    ax = axes[1, 4]

    ax.imshow(
        original_image,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    gt_overlay = np.zeros(
        (*ground_truth.shape, 4),
        dtype=np.float32
    )

    gt_overlay[ground_truth] = [0, 1, 0, 0.40]

    ax.imshow(gt_overlay)

    ax.set_title("Ground Truth Overlay", fontsize=10)
    ax.axis("off")

    axes[1, 0].set_ylabel(
        "SEGMENTATION",
        fontsize=12,
        fontweight="bold"
    )


    # -------------------------------------------------
    # LEGEND
    # -------------------------------------------------

    legend_items = [
        Patch(
            facecolor="red",
            alpha=0.40,
            label="Predicted lesion"
        ),
        Line2D(
            [0], [0],
            color="lime",
            linewidth=2,
            label="Ground-truth boundary"
        )
    ]

    fig.legend(
        handles=legend_items,
        loc="lower center",
        ncol=2,
        frameon=False,
        fontsize=10
    )

    fig.tight_layout(rect=[0, 0.06, 1, 0.92])

    # Save figure
    safe_label = case_label.lower().replace(" ", "_")

    output_path = (
        VISUALIZATION_DIR /
        f"{safe_label}_{filename.rsplit('.', 1)[0]}.png"
    )

    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight"
    )

    saved_figures.append(output_path)

    plt.show()
    plt.close(fig)


# -----------------------------------------------------
# VERIFY SAVED FIGURES
# -----------------------------------------------------

print("\n========== VISUALIZATION SUMMARY ==========")

print("Figures generated:", len(saved_figures))

for path in saved_figures:
    print(path.name, "| Exists:", path.exists())

assert len(saved_figures) == 3
assert all(path.exists() for path in saved_figures)

print("\nAll representative visualizations saved successfully.")

In [ ]:
# =====================================================
# SECTION 7.4: SELECT NORMAL IMAGE COMPARISON CASES
# =====================================================

normal_area_pivot = (
    results_df[results_df["category"] == "normal"]
    .pivot(
        index="filename",
        columns="condition",
        values="predicted_area"
    )
)

# Ensure a consistent condition order
normal_area_pivot = normal_area_pivot[
    ["original", "gaussian", "median", "non_local_means"]
]

normal_fp_pivot = normal_area_pivot > 0

# Summarize disagreement and predicted area
normal_case_summary = pd.DataFrame(index=normal_area_pivot.index)

normal_case_summary["models_with_false_positive"] = (
    normal_fp_pivot.sum(axis=1)
)

normal_case_summary["total_predicted_area"] = (
    normal_area_pivot.sum(axis=1)
)

normal_case_summary["maximum_predicted_area"] = (
    normal_area_pivot.max(axis=1)
)

normal_case_summary["area_difference"] = (
    normal_area_pivot.max(axis=1)
    - normal_area_pivot.min(axis=1)
)

# Select distinct images representing different behaviours
selected_normal_cases = []

selection_criteria = [
    ("Largest false-positive area", "maximum_predicted_area"),
    ("Largest difference between models", "area_difference"),
    ("Most model disagreement", "models_with_false_positive")
]

for label, column in selection_criteria:

    ranked = normal_case_summary.sort_values(
        column,
        ascending=False
    )

    for filename in ranked.index:

        if filename not in selected_normal_cases:
            selected_normal_cases.append(filename)
            break

# Add another distinct case with high total predicted area
for filename in normal_case_summary.sort_values(
    "total_predicted_area",
    ascending=False
).index:

    if filename not in selected_normal_cases:
        selected_normal_cases.append(filename)
        break

# Limit to four representative normal images
selected_normal_cases = selected_normal_cases[:4]

# Build comparison table
normal_visualization_df = normal_area_pivot.loc[
    selected_normal_cases
].reset_index()

normal_visualization_df["models_with_false_positive"] = (
    normal_visualization_df[
        ["original", "gaussian", "median", "non_local_means"]
    ].gt(0).sum(axis=1)
)

print("========== SELECTED NORMAL IMAGE CASES ==========")

display(normal_visualization_df)

print("\nSelected normal images:")
for filename in selected_normal_cases:
    print("-", filename)

# Save selected normal cases
normal_visualization_df.to_csv(
    RESULTS_DIR / "normal_visualization_cases.csv",
    index=False
)

print("\nNormal visualization cases saved successfully.")

In [ ]:
# =====================================================
# SECTION 7.5: RETRIEVE NORMAL IMAGE PREDICTIONS
# =====================================================

selected_normal_visuals = {
    filename: {
        "inputs": {},
        "predictions": {},
        "ground_truth": None
    }
    for filename in selected_normal_cases
}

# -----------------------------------------------------
# Retrieve normal images and predictions
# -----------------------------------------------------

for condition in comparison_conditions:

    print(f"\nRetrieving normal predictions: {condition}")

    model = models[condition]
    loader = test_loaders[condition]

    model.eval()

    with torch.no_grad():

        for images, masks, filenames, categories in loader:

            selected_indices = [
                i for i, filename in enumerate(filenames)
                if filename in selected_normal_visuals
            ]

            if not selected_indices:
                continue

            selected_batch = images[selected_indices].to(DEVICE)

            logits = model(selected_batch)
            probabilities = torch.sigmoid(logits)

            predictions = (
                probabilities >= THRESHOLD
            ).cpu().numpy()[:, 0].astype(bool)

            for output_idx, batch_idx in enumerate(selected_indices):

                filename = filenames[batch_idx]
                entry = selected_normal_visuals[filename]

                input_image = images[batch_idx, 0].numpy()
                ground_truth = (
                    masks[batch_idx, 0].numpy().astype(bool)
                )

                prediction = predictions[output_idx]

                assert categories[batch_idx] == "normal"
                assert ground_truth.sum() == 0, (
                    f"Expected empty mask for normal image: {filename}"
                )

                entry["inputs"][condition] = input_image
                entry["predictions"][condition] = prediction

                if entry["ground_truth"] is None:
                    entry["ground_truth"] = ground_truth
                else:
                    assert np.array_equal(
                        entry["ground_truth"],
                        ground_truth
                    )


# -----------------------------------------------------
# Verify retrieved data
# -----------------------------------------------------

print("\n========== NORMAL VISUALIZATION VERIFICATION ==========")

for filename, entry in selected_normal_visuals.items():

    assert set(entry["inputs"]) == set(comparison_conditions)
    assert set(entry["predictions"]) == set(comparison_conditions)
    assert entry["ground_truth"].sum() == 0

    print(f"\n{filename}")
    print("Ground-truth lesion pixels:", int(entry["ground_truth"].sum()))

    for condition in comparison_conditions:
        print(
            f"{condition:18s} | "
            f"Predicted pixels: "
            f"{int(entry['predictions'][condition].sum())}"
        )

print("\nAll normal images and predictions retrieved successfully.")


# -----------------------------------------------------
# Generate visualizations
# -----------------------------------------------------

normal_saved_figures = []

for filename, entry in selected_normal_visuals.items():

    original_image = entry["inputs"]["original"]

    fig, axes = plt.subplots(
        2, 5,
        figsize=(19, 8)
    )

    fig.suptitle(
        f"Normal Ultrasound: {filename}",
        fontsize=15,
        fontweight="bold"
    )

    # Top row: input images
    for col, condition in enumerate(comparison_conditions):

        ax = axes[0, col]

        ax.imshow(
            entry["inputs"][condition],
            cmap="gray",
            vmin=0,
            vmax=1
        )

        ax.set_title(
            condition_titles[condition],
            fontsize=11
        )

        ax.axis("off")

    axes[0, 4].imshow(
        entry["ground_truth"],
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[0, 4].set_title("Ground Truth (Empty)", fontsize=11)
    axes[0, 4].axis("off")

    axes[0, 0].set_ylabel(
        "INPUTS",
        fontsize=12,
        fontweight="bold"
    )

    # Bottom row: predictions over the same original image
    for col, condition in enumerate(comparison_conditions):

        ax = axes[1, col]

        prediction = entry["predictions"][condition]

        ax.imshow(
            original_image,
            cmap="gray",
            vmin=0,
            vmax=1
        )

        overlay = np.zeros(
            (*prediction.shape, 4),
            dtype=np.float32
        )

        overlay[prediction] = [1, 0, 0, 0.40]

        ax.imshow(overlay)

        predicted_area = int(prediction.sum())

        ax.set_title(
            f"{condition_titles[condition]}\n"
            f"False-positive pixels: {predicted_area}",
            fontsize=10
        )

        ax.axis("off")

    # Empty ground-truth overlay
    axes[1, 4].imshow(
        original_image,
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[1, 4].set_title(
        "Ground Truth Overlay\nNo lesion",
        fontsize=10
    )

    axes[1, 4].axis("off")

    axes[1, 0].set_ylabel(
        "PREDICTIONS",
        fontsize=12,
        fontweight="bold"
    )

    fig.legend(
        handles=[
            Patch(
                facecolor="red",
                alpha=0.40,
                label="Predicted false-positive region"
            )
        ],
        loc="lower center",
        frameon=False,
        fontsize=10
    )

    fig.tight_layout(rect=[0, 0.06, 1, 0.92])

    output_path = (
        VISUALIZATION_DIR /
        f"normal_{filename.rsplit('.', 1)[0]}.png"
    )

    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight"
    )

    normal_saved_figures.append(output_path)

    plt.show()
    plt.close(fig)


print("\n========== NORMAL VISUALIZATION SUMMARY ==========")

print("Figures generated:", len(normal_saved_figures))

for path in normal_saved_figures:
    print(path.name, "| Exists:", path.exists())

assert len(normal_saved_figures) == len(selected_normal_cases)
assert all(path.exists() for path in normal_saved_figures)

print("\nAll normal-image visualizations saved successfully.")

## 8. Final Comparative Results and Graphs

Visual comparison of the four U-Net models using:
- Mean Dice and IoU
- Pixel-level precision and sensitivity
- Benign and malignant lesion performance
- False-positive behaviour on normal images
- Paired Dice improvements with bootstrap confidence intervals

In [ ]:
# =====================================================
# SECTION 8.1: FINAL COMPARATIVE GRAPHS
# =====================================================

PLOT_DIR = VISUALIZATION_DIR

plot_order = [
    "original",
    "gaussian",
    "median",
    "non_local_means"
]

plot_labels = [
    "Original",
    "Gaussian",
    "Median",
    "NLM"
]


# -----------------------------------------------------
# GRAPH 1: OVERALL DICE AND IOU
# -----------------------------------------------------

plot_data = overall_summary_df.set_index(
    "condition"
).loc[plot_order]

x = np.arange(len(plot_order))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(
    x - width / 2,
    plot_data["mean_dice"],
    width,
    label="Mean Dice"
)

ax.bar(
    x + width / 2,
    plot_data["mean_iou"],
    width,
    label="Mean IoU"
)

ax.set_title(
    "Overall Lesion Segmentation Performance",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylabel("Score")
ax.set_xlabel("Image processing condition")
ax.set_xticks(x)
ax.set_xticklabels(plot_labels)

ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()

fig.savefig(
    PLOT_DIR / "overall_dice_iou_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# -----------------------------------------------------
# GRAPH 2: PIXEL-LEVEL PRECISION AND SENSITIVITY
# -----------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6))

ax.bar(
    x - width / 2,
    plot_data["pixel_precision"],
    width,
    label="Pixel Precision"
)

ax.bar(
    x + width / 2,
    plot_data["pixel_sensitivity"],
    width,
    label="Pixel Sensitivity"
)

ax.set_title(
    "Pixel-Level Performance Comparison",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylabel("Score")
ax.set_xlabel("Image processing condition")
ax.set_xticks(x)
ax.set_xticklabels(plot_labels)

ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()

fig.savefig(
    PLOT_DIR / "pixel_precision_sensitivity_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# -----------------------------------------------------
# GRAPH 3: BENIGN AND MALIGNANT DICE
# -----------------------------------------------------

fig, ax = plt.subplots(figsize=(11, 6))

category_order = ["benign", "malignant"]
category_x = np.arange(len(category_order))

width_category = 0.18

for i, condition in enumerate(plot_order):

    subset = category_summary_df[
        category_summary_df["condition"] == condition
    ].set_index("category").loc[category_order]

    offsets = (
        i - (len(plot_order) - 1) / 2
    ) * width_category

    ax.bar(
        category_x + offsets,
        subset["mean_dice"],
        width_category,
        label=plot_labels[i]
    )

ax.set_title(
    "Category-Wise Lesion Segmentation Performance",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Lesion category")
ax.set_ylabel("Mean Dice")

ax.set_xticks(category_x)
ax.set_xticklabels(["Benign", "Malignant"])

ax.set_ylim(0, 1)

ax.legend(title="Condition")
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()

fig.savefig(
    PLOT_DIR / "category_wise_dice_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# -----------------------------------------------------
# GRAPH 4: FALSE-POSITIVE NORMAL IMAGES
# -----------------------------------------------------

normal_plot = normal_summary_df.set_index(
    "condition"
).loc[plot_order]

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    plot_labels,
    normal_plot["false_positive_images"]
)

ax.set_title(
    "False-Positive Images Among Normal Ultrasounds",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Image processing condition")
ax.set_ylabel("Number of false-positive images")

ax.set_ylim(0, 21)

ax.grid(axis="y", alpha=0.25)

ax.bar_label(bars, padding=3)

fig.tight_layout()

fig.savefig(
    PLOT_DIR / "normal_false_positive_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# -----------------------------------------------------
# GRAPH 5: MEAN FALSE-POSITIVE AREA ON NORMAL IMAGES
# -----------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    plot_labels,
    normal_plot["mean_predicted_area"]
)

ax.set_title(
    "Mean Predicted Lesion Area in Normal Images",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Image processing condition")
ax.set_ylabel("Mean predicted pixels per normal image")

ax.grid(axis="y", alpha=0.25)

ax.bar_label(bars, padding=3, fmt="%.1f")

fig.tight_layout()

fig.savefig(
    PLOT_DIR / "normal_false_positive_area_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# -----------------------------------------------------
# GRAPH 6: PAIRED DICE IMPROVEMENT WITH 95% CI
# -----------------------------------------------------

dice_stats = statistical_summary_df[
    statistical_summary_df["metric"] == "dice"
].copy()

condition_order = [
    "gaussian",
    "median",
    "non_local_means"
]

dice_stats = (
    dice_stats.set_index("condition")
    .loc[condition_order]
    .reset_index()
)

y_values = dice_stats["mean_difference"].to_numpy()

lower_errors = (
    y_values - dice_stats["ci_lower"].to_numpy()
)

upper_errors = (
    dice_stats["ci_upper"].to_numpy() - y_values
)

fig, ax = plt.subplots(figsize=(10, 6))

positions = np.arange(len(condition_order))

ax.errorbar(
    positions,
    y_values,
    yerr=np.vstack([lower_errors, upper_errors]),
    fmt="o",
    capsize=6,
    markersize=8
)

ax.axhline(
    0,
    linestyle="--",
    linewidth=1,
    label="No difference from original"
)

ax.set_title(
    "Paired Mean Dice Improvement Over Original U-Net",
    fontsize=14,
    fontweight="bold"
)

ax.set_ylabel("Mean Dice difference")
ax.set_xlabel("Filtering method")

ax.set_xticks(positions)
ax.set_xticklabels(["Gaussian", "Median", "NLM"])

ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.tight_layout()

fig.savefig(
    PLOT_DIR / "paired_dice_improvement_confidence_intervals.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)


# -----------------------------------------------------
# VERIFY ALL SAVED GRAPHS
# -----------------------------------------------------

expected_graphs = [
    "overall_dice_iou_comparison.png",
    "pixel_precision_sensitivity_comparison.png",
    "category_wise_dice_comparison.png",
    "normal_false_positive_comparison.png",
    "normal_false_positive_area_comparison.png",
    "paired_dice_improvement_confidence_intervals.png"
]

print("\n========== FINAL GRAPH SUMMARY ==========")

for filename in expected_graphs:

    path = PLOT_DIR / filename

    print(
        filename,
        "| Exists:",
        path.exists(),
        "| Size:",
        f"{path.stat().st_size / 1024:.1f} KB" if path.exists() else "Missing"
    )

assert all(
    (PLOT_DIR / filename).exists()
    for filename in expected_graphs
)

print("\nAll six comparative graphs saved successfully.")

In [ ]:
# =====================================================
# SECTION 9.1: FINAL COMPARISON TABLE
# =====================================================

# 1. Overall lesion performance
final_comparison_df = overall_summary_df[
    [
        "condition",
        "lesion_images",
        "mean_dice",
        "mean_iou",
        "pixel_precision",
        "pixel_sensitivity"
    ]
].copy()

# 2. Category-wise Dice
category_pivot = category_summary_df.pivot(
    index="condition",
    columns="category",
    values="mean_dice"
).reset_index()

category_pivot = category_pivot.rename(columns={
    "benign": "benign_mean_dice",
    "malignant": "malignant_mean_dice"
})

final_comparison_df = final_comparison_df.merge(
    category_pivot,
    on="condition",
    validate="one_to_one"
)

# 3. Normal-image performance
normal_columns = normal_summary_df[
    [
        "condition",
        "false_positive_images",
        "false_positive_rate",
        "mean_predicted_area"
    ]
].rename(columns={
    "false_positive_images": "normal_false_positive_images",
    "false_positive_rate": "normal_false_positive_rate",
    "mean_predicted_area": "normal_mean_predicted_area"
})

final_comparison_df = final_comparison_df.merge(
    normal_columns,
    on="condition",
    validate="one_to_one"
)

# 4. Dice change relative to original control
original_dice = final_comparison_df.loc[
    final_comparison_df["condition"] == "original",
    "mean_dice"
].iloc[0]

final_comparison_df["dice_change_vs_original"] = (
    final_comparison_df["mean_dice"] - original_dice
)

# 5. Sort by mean Dice for presentation
final_comparison_df = final_comparison_df.sort_values(
    "mean_dice",
    ascending=False
).reset_index(drop=True)

# 6. Save
final_comparison_path = (
    RESULTS_DIR / "final_comparison_table.csv"
)

final_comparison_df.to_csv(
    final_comparison_path,
    index=False
)

print("========== FINAL COMPARISON TABLE ==========")

display(final_comparison_df.round(4))

print("\nSaved:", final_comparison_path)
print("\nRows:", len(final_comparison_df))
print("Conditions:", final_comparison_df["condition"].tolist())

assert len(final_comparison_df) == 4
assert final_comparison_df["condition"].nunique() == 4

print("\nFinal comparison table created successfully.")

In [ ]:
# =====================================================
# SECTION 9.2: ADD STATISTICAL RESULTS TO FINAL TABLE
# =====================================================

# Extract Dice statistical comparisons
dice_stats = statistical_summary_df[
    statistical_summary_df["metric"] == "dice"
][
    [
        "condition",
        "mean_difference",
        "ci_lower",
        "ci_upper",
        "p_value",
        "p_value_holm"
    ]
].copy()

dice_stats = dice_stats.rename(columns={
    "mean_difference": "paired_dice_difference",
    "ci_lower": "dice_ci95_lower",
    "ci_upper": "dice_ci95_upper",
    "p_value": "dice_wilcoxon_p",
    "p_value_holm": "dice_wilcoxon_p_holm"
})

# Merge with the master comparison table
final_comparison_df = final_comparison_df.merge(
    dice_stats,
    on="condition",
    how="left",
    validate="one_to_one"
)

# Save the enhanced table
final_comparison_df.to_csv(
    RESULTS_DIR / "final_comparison_table.csv",
    index=False
)

print("========== ENHANCED FINAL COMPARISON ==========")

display(final_comparison_df.round(5))

print("\nFinal table updated with statistical evidence.")
print("Saved:", RESULTS_DIR / "final_comparison_table.csv")

# 10. Conclusions and Discussion

## 10.1 Research Objective

This study investigated whether despeckling ultrasound images before U-Net-based breast lesion segmentation improves segmentation performance.

Three filtering approaches were evaluated:
- Gaussian filtering
- Median filtering
- Non-Local Means (NLM) filtering

The results were compared against an original-image U-Net model without additional filtering.

All four models were evaluated on the same independent test set containing 118 images, including 97 lesion images (67 benign and 30 malignant) and 21 normal images.

## 10.2 Main Findings

### Overall Segmentation Performance

Median filtering achieved the highest mean Dice coefficient (0.6565) and mean IoU (0.5492), followed by Gaussian filtering, NLM, and the original-image model.

Compared with the original-image model, mean Dice improved by:

- Median filtering: +0.0337
- Gaussian filtering: +0.0288
- NLM filtering: +0.0099

These results indicate that median and Gaussian filtering are promising approaches for improving average lesion overlap in this experiment.

### Category-Wise Performance

Median filtering achieved the highest mean Dice for benign lesions (0.6967).

Gaussian filtering achieved the highest mean Dice for malignant lesions (0.5688), closely followed by median filtering (0.5665).

All four models performed better on benign lesions than malignant lesions.

### Pixel-Level Performance

The original-image model achieved the highest pooled pixel precision (0.8548).

Median filtering achieved the highest pixel sensitivity (0.5750), indicating that it recovered a larger proportion of ground-truth lesion pixels.

The filtered models showed a general trade-off between precision and sensitivity: increased sensitivity was accompanied by additional false-positive pixels on lesion images.

### Normal-Image False Positives

Gaussian filtering produced false-positive predictions on 10 of 21 normal images, compared with:

- Original: 12 of 21
- Median: 15 of 21
- NLM: 13 of 21

NLM produced the lowest mean predicted lesion area on normal images, while median filtering produced the highest.

These results suggest that filtering does not consistently reduce false-positive predictions.

## 10.3 Statistical Analysis

Paired comparisons were performed across the same 97 lesion images.

Mean Dice improvements over the original model were:

- Gaussian: +0.0288
- Median: +0.0337
- NLM: +0.0099

Paired bootstrap confidence intervals and Wilcoxon signed-rank tests were used to assess the differences.

After Holm correction, none of the three filtering methods showed a statistically significant Dice improvement at the 0.05 significance level.

Although median filtering achieved the largest average improvement, its adjusted p-value was 0.1610.

Therefore, the observed improvements should be interpreted as promising experimental trends rather than conclusive evidence that despeckling improves segmentation.

## 10.4 Overall Conclusion

The experiment demonstrated that preprocessing ultrasound images using despeckling filters can influence the performance of U-Net-based breast lesion segmentation.

Among the evaluated filters, median filtering achieved the highest average Dice and IoU scores, while Gaussian filtering provided the best malignant-lesion Dice and the lowest false-positive image rate on normal cases.

However, none of the filtering methods demonstrated a statistically significant Dice improvement after multiple-comparison correction.

The findings indicate that the effect of despeckling is dependent on the evaluation metric and lesion category. Improving lesion overlap does not necessarily translate into fewer false-positive predictions on normal images.

Consequently, no single filtering method can be considered universally superior based on the current experiment.

## 10.5 Limitations

1. **Single dataset:** The experiment used the BUSI breast ultrasound dataset. Results may not generalize to other ultrasound datasets or clinical environments.

2. **Single random seed:** All four models were trained using seed 42. Additional independent training runs are required to assess variability due to model initialization and training randomness.

3. **Limited test-set size:** The independent test set contained 97 lesion images and 21 normal images, limiting the strength and generalizability of statistical conclusions.

4. **Limited filtering approaches:** Only Gaussian, median and NLM filtering were investigated. Other despeckling techniques were not included.

5. **Fixed filter parameters:** The study evaluated the selected filter configurations rather than systematically optimizing filter parameters.

6. **Segmentation-focused evaluation:** The study evaluated mask overlap and pixel-level performance, not diagnostic accuracy, clinical outcomes or radiologist agreement.

7. **Checkpoint variation:** The earlier baseline checkpoint was trained separately from the four checkpoints used in the main comparison. Therefore, the original-condition checkpoint from the comparative training experiment was retained as the control for the primary analysis.

## 10.6 Future Work

Future investigations could include:

- Training multiple independent seeds for each filtering condition.
- Exploring additional filters and adaptive despeckling methods.
- Investigating the effect of filtering strength on lesion boundaries.
- Evaluating performance on an external ultrasound dataset.
- Studying whether filter selection based on image characteristics can improve segmentation consistency.
- Examining boundary-specific metrics in addition to Dice and IoU.

The overall objective of future work should be to determine whether despeckling can improve segmentation reliability without removing diagnostically relevant boundary information.